# train ResNet18 + baselines + freeze metrics


Script entry points under `ai/scripts/`:

1. **Freeze data** + **smoke** (1 epoch, small subset) → `models/smoke_checkpoint.pt`
2. **Full train** ResNet18 → `models/best_model.pt` (Colab GPU)
3. **Baseline** Histogram + FFT → `models/baseline_hist_fft.json`
4. **Evaluate** ROC / CI / JPEG held-out / Grad-CAM → `metrics/metrics.json`

JPEG stays `held_out=True` and is never used in training.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

os.chdir(Path("..").resolve())
print("cwd", Path.cwd())
Path("models").mkdir(exist_ok=True)
Path("metrics").mkdir(exist_ok=True)

## 1) Data freeze + smoke test (1 epoch)

Writes `metadata/data_freeze.json` and checks that the training loop runs.

In [ ]:
subprocess.check_call([sys.executable, "scripts/train_resnet.py", "--smoke", "--batch-size", "16"])

## 2) Full ResNet18 training

Uses all usable (non-JPEG) train/val rows. Best validation checkpoint → `models/best_model.pt`.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Full training on CPU is too slow. "
        "Open notebooks/colab_train.ipynb on Google Colab with a GPU instead."
    )

subprocess.check_call([
    sys.executable, "scripts/train_resnet.py",
    "--epochs", "8",
    "--batch-size", "64",
    "--lr", "1e-4",
])

## 3) Histogram + FFT baseline (research comparison)

In [ ]:
subprocess.check_call([sys.executable, "scripts/baseline_hist_fft.py"])

## 4) Freeze metrics (ROC, CI, JPEG held-out, Grad-CAM)

In [ ]:
subprocess.check_call([sys.executable, "scripts/evaluate.py", "--checkpoint", "models/best_model.pt"])

import json
metrics = json.loads(Path("metrics/metrics.json").read_text(encoding="utf-8"))
metrics["test_overall"], metrics["jpeg_held_out"]